In [10]:
import math
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.patches import Ellipse
from itertools import chain
import time
from tqdm import tqdm
from operator import attrgetter
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from scipy.stats import chi2
from scipy.spatial.distance import pdist

# surface functions, merge function, and other helper functions

def surface_single(x, U, A, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for a single point.

    Parameters
    ----------
    x : np.ndarray, shape (d,)
        The query point.
    U : np.ndarray, shape (d, d)
        Eigenvectors (columns) defining the ellipsoid orientation.
    A : np.ndarray, shape (d,)
        Semi-axis lengths along each eigenvector direction.
    p : float or np.ndarray, shape (d)
        power/exponent of the ellipsoid axes.
    k : float
        confidence value

    Returns
    -------
    float
        < 0 if x is inside the ellipsoid
        = 0 if x is on the surface
        > 0 if x is outside the ellipsoid
    """
    x_local = U.T @ x        # Project into ellipsoid's local frame, shape (d,)
    return np.sum(np.abs(x_local / A) ** p) - (k ** 2)

def surface_multi(xs, Us, As, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for n points,
    each against its own ellipsoid.

    Parameters
    ----------
    xs : np.ndarray, shape (n, d)
        Query points.
    Us : np.ndarray, shape (n, d, d)
        Per-ellipsoid eigenvector matrices.
    As : np.ndarray, shape (n, d)
        Per-ellipsoid semi-axis lengths.

    Returns
    -------
    np.ndarray, shape (n,)
        Surface function value for each (point, ellipsoid) pair.
        < 0 inside, = 0 on surface, > 0 outside.
    """
    xs_local = np.einsum('ndi, ni -> nd', Us.swapaxes(1,2), xs)  # (n, d)
    return np.sum(np.abs(xs_local / As) ** p, axis=1) - (k ** 2)              # (n,)

def merge_nodes(x, y): # NOTE: New P is currently avg of the 2 that is (x.p + y.p) / 2
    new_n = x.n + y.n
    dist = x.M - y.M
    new_M = ((x.n * x.M) + (y.n * y.M)) / (new_n)
    new_S = ((x.n / new_n) * x.S) + ((y.n / new_n) * y.S) + (((x.n * y.n)/ (new_n ** 2)) * (np.outer(dist, dist)))
    eigen_value, eigen_vector = np.linalg.eigh(new_S)
    # reverse the order of the eigenvalue/vectors to be in DESCENDING order
    eigen_value = eigen_value[::-1]
    eigen_vector = eigen_vector[:, ::-1]
    # eigen_vector = eigen_vector.T
    # calculate new width that based on confidence ellipsoid
    confidence = new_n / (new_n + x.dim)
    chi = chi2.ppf(confidence, x.dim)
    new_A = np.sqrt(np.abs(eigen_value) * chi)
    new_A[new_A == 0] = x.eps

    return Node(dimension=x.dim, label=x.label, A=new_A, n=new_n, M=new_M, S=new_S, U=eigen_vector, eps=x.eps, alpha=x.alpha, p=(x.p + y.p) / 2)

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]

class Node():
    def __init__(self,
        dimension, label,
        A, p, eps, alpha,
        n=1,
        M=None,
        S=None,
        U=None,
        boundary=None):
        """
        A node contain information about a single ellipsoid such as its centre, cov matrix, and the axis length andd direction.
        It can be "updated" with a point to move it to learn and cover more data point.
        
        Parameters
        A (np.ndarray) (dim) : The length of each axis. Not necessary sorted.  np array 
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function.
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        n (int) : The number of data points this node has learnt from.
        U (np.ndarray) (dim, dim) : The eigenvectors of the cov matrix. NOT transposed that is U[i] contain the ith eigenvector.
        M (np.ndarray) (dim) : The mean/center of the ellipsoid
        S (np.ndarray) (dim, dim) : The cov matrix of the ellipsoid. 
        boundary (dict) : A dictionary containing the boundary point and a special key, _max containing the maximum value.
        max_boundary (int) : The maximum number of points allowed in the boundary.
        """

        self.dim = dimension
        self.label = label
        self.A = A
        self.p = p
        self.eps = eps
        self.alpha = alpha
        self.n = n
        self.confidence = n / (n + self.dim)
        self.chi = chi2.ppf(self.confidence, self.dim)

        self.U = np.eye(dimension) if U is None else U
        self.M = np.array([0.] * dimension) if M is None else M
        self.S = np.zeros((dimension, dimension)) if S is None else S

        if boundary is not None:
            self.boundary = boundary
        else:
            self.boundary = []
        self.max_boundary = 2 * dimension

    def update_boundary(self, x, xd):
        """
        Update the boundary that contain the points nearest to the edge.
        This function is called when the ellipsoid is moved/updated.
        The point x is also considered whether it is a boundary point.
        """
        #print(x, self.boundary)
        # First, update the points in the boundary because the ellipsoid might have moved.
        for p in self.boundary:
            d = surface_single(p - self.M, self.U, self.A, self.p)

            if d > 0:
                for i, x in enumerate(self.boundary):
                    if np.allclose(x, p):
                        self.boundary.pop(i)
                        break

        # If the number of points in the boundary is small we simply add the new point
        d = surface_single(x - self.M, self.U, self.A, self.p)
        if d > 0:
            return None
        if len(self.boundary) < self.max_boundary:
            self.boundary.append(x)
            return None

        # Create copies of the boundary
        A = np.array(self.boundary)
        As = np.repeat(A[None, :, :], A.shape[0], axis=0)
        # Replace one row in each copy of A with x
        As[np.arange(A.shape[0]), np.arange(A.shape[0])] = x

        def f(mats):
            # mats has shape (k, n, m)
            gram = np.matmul(mats.transpose(0, 2, 1), mats)  # shape (k, m, m)
            return np.linalg.det(gram)
        det = f(np.array([A]))
        dets = f(As)

        if det > dets.max():
            pass
        else:
            self.boundary[np.argmax(dets)] = x

        """# Else we remove the point closest to the center (this might be x),        
        if d > self.boundary["_min"][1]:
            self.boundary.pop(self.boundary["_min"][0])
            self.boundary[tuple(x)] = d
            if d > self.boundary["_max"][1]:
                self.boundary["_max"] = (p, d)
            if d < self.boundary["_min"][1]:
                self.boundary["_min"] = (p, d)"""

    def update_exponent(self):
        if len(self.boundary) < self.max_boundary:
            return None
        print("pass")
        x0 = self.p
        cons = []
        for b_point in self.boundary:
            def con(x):
                return - surface_single(b_point - self.M, self.U, self.A, x)
            cons.append({'type': 'ineq', 'fun': con})
            
        solution = minimize(np.sum, x0, method='SLSQP', constraints=cons, bounds=[(1, 10) for _ in range(len(self.p))])
        print("replace",self.p,solution.x)
        self.p = solution.x

    def update(self, x, parent):
        """
        Update the node with a new data point x.
        Performing the following steps.
        1. Calculate the new mean and cov matrix of the node including the new point
        2. Calculate the eigenvalue/vector of the new cov matrix
        3. Update the width of the axes of the ellipsoid
        4. Calculate the growth criterion to check whether the updated ellipsoid cover the new data point
        5. If the growth criterion is acceptable the node is allowed to grow/update to include this new data point,
           else the node is not updated and a new node is added to cover the new datapoint instead.

        Parameter
        x (np.ndarray) (dim) : A new data point the model need to learn/update from.
        parent (VEBF) : The VEBF model this node belong to. We only need this in case we need to add a new node to the VEBF model. 
        """

        # Calculate new mean and cov matrix
        n = self.n
        M = self.M
        alpha = n / (n + 1)
        beta = x / (n + 1)
        M_new = (alpha * M) + beta
        k_1 = (np.outer(x, x)/(n + 1)) - np.outer(M_new, M_new) + np.outer(M, M)
        k_2 = - (np.outer(M, M) / (n + 1))
        k = k_1 + k_2
        S_new = (alpha * self.S) + k

        eigen_value, eigen_vector = np.linalg.eigh(S_new)
        # reverse the order of the eigenvalue/vectors to be in DESCENDING order
        eigen_value = eigen_value[::-1]
        eigen_vector = eigen_vector[:, ::-1]

        # calculate new width with adaptively by combining fixed and dynamic width update rules
        """beta = 1 - self.alpha
        fixed = np.sqrt(np.pi * 2 * np.abs(eigen_value))
        dynamic = self.A + ((M_new - self.M) @ eigen_vector.T)
        new_A = (self.alpha * fixed) + (beta * dynamic)
        new_A[new_A == 0] = self.eps"""

        # calculate new width that based on confidence ellipsoid
        new_A = np.sqrt(np.abs(eigen_value) * self.chi)
        new_A[new_A == 0] = self.eps
        
        gc = surface_single(x - M_new, eigen_vector, self.A, self.p)
        if gc <= 0:
            # update the current node
            self.M = M_new
            self.U = eigen_vector
            self.S = S_new
            self.A = new_A
            self.n += 1
            self.confidence = n / (n + self.dim)
            self.chi = chi2.ppf(self.confidence, self.dim)
            # self.update_boundary(x, gc)
            # self.update_exponent()
            return self
        else:
            # add new node
            new_node = Node(self.dim, self.label, M=x, eps=self.eps, A=parent.default_width, alpha=self.alpha, p=self.p)
            parent.nodes[self.label].append(new_node)
            return new_node

    def __str__(self):
        return f"A node covering {self.n} points."

class VEBF():
    def __init__(self, dimension, merge_parameter=0, eps=0.00001, default_width=None, alpha=0.55, p=None):
        """
        The Versatile Ellipsoid Basis Function model.
        Geometrically, the model learn by covering the datapoints with ellipsoids to learn the distribution of the data.
        
        Parameter
        dimension (int) : Number of dimension of input feature.
        merge_parameter (float) : The hyperparameter that controll when two ellipsoids/nodes will be merged.
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function. (dim) np array
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        default_width (np.ndarray) (dim) : The default axis width of the ellipsoid.
        """
        self.dim = dimension
        self.merge_parameter = merge_parameter
        self.nodes = {}
        self.eps = eps
        self.alpha = alpha
        self.default_width = np.array([1.] * dimension) if default_width is None else default_width
        self.p = np.array([2.] * dimension) if p is None else p # default to "normal" hyperellipsoid with p=2

    def find_shortest(self, x, y):
        # TODO: vectorize this
        min_dist = float('inf')
        min_node = None
        for node in self.nodes[y]:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    # NOTE: can combine this with find_shortest by using y=None as default value
    def find_shortest_all(self, x, nodes=None):
        min_dist = float('inf')
        min_node = None
        nodes = self.get_nodes() if nodes is None else nodes
        for node in nodes:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    def check_merge(self, x):
        """
        Check for pair of nodes which can be merged.
        Only consider the "latest" node, x

        Parameters:
        x (Node): The node which has just been updated or created.
        """
        nodes = self.nodes[x.label]
        if len(nodes) == 1:
            return
        nodes.remove(x)
        
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])
        
        scores = surface_multi(x.M - Ms, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(x, y)
            nodes.append(new_node)
            self.check_merge(new_node)
            return

        n = len(nodes)
        U = x.U
        A = x.A
        P = x.p
        Us = np.broadcast_to(U, (n,) + U.shape)
        As = np.broadcast_to(A, (n,) + A.shape)
        Ps = np.broadcast_to(P, (n,) + P.shape)

        scores = surface_multi(Ms - x.M, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(y, x)
            nodes.append(new_node)
            self.check_merge(new_node)
            return
            
        nodes.append(x)

    def get_nodes(self):
        return list(chain.from_iterable(self.nodes.values()))

    def predict(self, x):

        nodes = self.get_nodes()
        assert len(nodes) > 0, "Can't predict with empty VEBF."
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])

        scores = surface_multi(x - Ms, Us, As, Ps)
        idx = np.argsort(scores)
        result = nodes[idx[0]].label
        return result
        """ OLD prediction by euclidean

        candidates = [node for node, score in zip(nodes, scores) if score <= 0]

        if len(candidates) == 0:
            closest = self.find_shortest_all(x)
            return closest.label
        return self.find_shortest_all(x, candidates).label"""
            
    def train(self, x, y):
        if y not in self.nodes: # Add a new node for unseen label.
            self.nodes[y] = [Node(dimension=self.dim, label=y, M=x, eps=self.eps, A=self.default_width, alpha=self.alpha, p=self.p)]
        else: # Update closest existing node for existing label.
            shortest = self.find_shortest(x, y)
            latest = shortest.update(x, self) # latest is the node with latest change, which need to checked for merge
            self.check_merge(latest)

    def __str__(self):
        total = sum(len(x) for x in self.nodes.values())
        string = f"A vebf with {total} nodes.\n"
        for label, nodes in self.nodes.items():
            for node in nodes:
                string += f"Label {label}: {str(node)}\n"
        return string

In [11]:
# Loading the datasets

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]
    
with open("data/iris/iris.data", "r") as f:
    iris_data = f.readlines()
iris_ds = []
for line in iris_data[:-1]:
    line = line.strip().split(',')
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    iris_ds.append((x, y))

with open("data/ecoli/ecoli.data", "r") as f:
    ecoli_data = f.readlines()
ecoli_ds = []
for line in ecoli_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    ecoli_ds.append((x, y))

with open("data/image_seg/segmentation.data", "r") as f:
    seg_train_data = f.readlines()
with open("data/image_seg/segmentation.test", "r") as f:
    seg_test_data = f.readlines()
seg_train = []
for line in seg_train_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_train.append((x, y))
seg_test = []
for line in seg_test_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_test.append((x, y))

with open("data/waveform/waveform.data", "r") as f:
    waveform_data = f.readlines()
wave_ds = []
for line in waveform_data:
    line = line.strip().split(",")
    y = line[-1]
    x = np.array(line[:-1], dtype=float)
    wave_ds.append((x, y))

with open("data/yeast/yeast.data", "r") as f:
    yeast_data = f.readlines()
yeast_ds = []
for line in yeast_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    yeast_ds.append((x, y))

with open("data/anuran/Frogs_MFCCs.csv", "r") as f:
    anuran_data = f.readlines()
anuran_ds = []
for line in anuran_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[:-4], dtype=float)
    y = tuple(line[-4:-1])
    anuran_ds.append((x, y))

with open("data/spambase/spambase.data", "r") as f:
    spam_data = f.readlines()
spam_ds = []
for line in spam_data:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    spam_ds.append((x, y))

with open("data/letter/letter-recognition.data", "r") as f:
    letter_data = f.readlines()
letter_ds = []
for line in letter_data:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    letter_ds.append((x, y))

with open("data/bankrupt/data.csv", "r") as f:
    bankrupt_data = f.readlines()
bankrupt_ds = []
for line in bankrupt_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    bankrupt_ds.append((x, y))

with open("data/digits/optdigits.tes", "r") as f:
    digits_test_data = f.readlines()
digits_test = []
for line in digits_test_data[:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    digits_test.append((x, y))

with open("data/digits/optdigits.tra", "r") as f:
    digits_train_data = f.readlines()
digits_train = []
for line in digits_train_data[:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    digits_train.append((x, y))

with open("data/phishing/PhishingData.arff", "r") as f:
    phishing_data = f.readlines()
phishing_ds = []
for line in phishing_data[14:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    phishing_ds.append((x, y))


In [12]:
def initial_width(ds, delta=1):
    avg_dist = pdist([x for x,_ in ds], metric="euclidean").mean()
    return avg_dist

iris_dim = len(iris_ds[0][0])
anuran_dim = len(anuran_ds[0][0])
spam_dim = len(spam_ds[0][0])
seg_dim = len(seg_train[0][0])
wave_dim = len(wave_ds[0][0])
letter_dim = len(letter_ds[0][0])
yeast_dim = len(yeast_ds[0][0])
digits_dim = len(digits_train[0][0])
bankrupt_dim = len(bankrupt_ds[0][0])
phishing_dim = len(phishing_ds[0][0])

iris_A = initial_width(iris_ds)
anuran_A = initial_width(anuran_ds)
spam_A = initial_width(spam_ds)
seg_A = initial_width(seg_train)
wave_A = initial_width(wave_ds)
letter_A = initial_width(letter_ds)
yeast_A = initial_width(yeast_ds)
digits_A = initial_width(digits_train)
bankrupt_A = initial_width(bankrupt_ds)
phishing_A = initial_width(phishing_ds)
# The implementation of confidence based VEBF takes array of width as initial width
iris_A = np.array([iris_A] * iris_dim)
anuran_A = np.array([anuran_A] * anuran_dim)
spam_A = np.array([spam_A] * spam_dim)
seg_A = np.array([seg_A] * seg_dim)
wave_A = np.array([wave_A] * wave_dim)
letter_A = np.array([letter_A] * letter_dim)
yeast_A = np.array([yeast_A] * yeast_dim)
digits_A = np.array([digits_A] * digits_dim)
bankrupt_A = np.array([bankrupt_A] * bankrupt_dim)
phishing_A = np.array([phishing_A] * phishing_dim)




In [13]:
def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))

In [14]:
for delta in [1, 2, 4, 8, 16]:

    seg_result = []
    for i in tqdm(range(30)):

        # shuffle the train/test split
        random.shuffle(seg_train)
        model = VEBF(dimension=seg_dim, merge_parameter=0., default_width=seg_A * delta)
        for idx in range(len(seg_train)):
            x,y = seg_train[idx]
            x = np.array(x,dtype=float)
            model.train(x, y)

        # test the model
        correct = 0
        for x, label in seg_test:
            pred = model.predict(x)
            if pred == label:
                correct += 1
        
        seg_result.append(correct/len(seg_test))
    print("delta:", delta)
    print_stat(seg_result, "seg")
    print()

100%|██████████| 30/30 [00:01<00:00, 16.15it/s]


delta: 1
seg Mean: 0.6268571428571429
seg Max: 0.7561904761904762
seg Min: 0.36142857142857143



100%|██████████| 30/30 [00:01<00:00, 17.46it/s]


delta: 2
seg Mean: 0.7876984126984128
seg Max: 0.8257142857142857
seg Min: 0.7442857142857143



100%|██████████| 30/30 [00:01<00:00, 17.28it/s]


delta: 4
seg Mean: 0.6734444444444444
seg Max: 0.7276190476190476
seg Min: 0.580952380952381



100%|██████████| 30/30 [00:01<00:00, 17.22it/s]


delta: 8
seg Mean: 0.7697301587301589
seg Max: 0.8485714285714285
seg Min: 0.34714285714285714



100%|██████████| 30/30 [00:01<00:00, 16.17it/s]

delta: 16
seg Mean: 0.8080952380952382
seg Max: 0.8461904761904762
seg Min: 0.7495238095238095



In [15]:
"""Reference-style implementation of Versatile Elliptic Basis Function (VEBF).

This version follows the VEBF rules used by the reference implementation:
- Initial node width: one global average pairwise distance vector per training batch.
- New node: center at the incoming sample, zero covariance, identity eigenvectors,
  n = 1, and the class's initialized width.
- Growth test: tentative center + tentative covariance/eigenvectors are used with
  the current node width.
- Width update: only after n > N0 (N0 = 2), expand each axis by the projected
  center displacement.
- Merge: merge same-class nodes when either center is inside the other's
  hyperellipsoid (psi <= theta); merged width is sqrt(2*pi*|lambda|).
- Prediction: minimum hyperellipsoidal function value over all nodes.

The public train(x, y) method is retained as a one-sample convenience wrapper.
For paper-style initialization, prefer fit(X, y) for the initial batch and then
partial_fit(X_chunk, y_chunk) for subsequent streaming updates.
"""

import math
import random
from itertools import chain

import numpy as np
from scipy.spatial.distance import pdist


class Node:
    """A VEBF hyperellipsoidal node."""

    def __init__(self, dimension, label, A, n=1, M=None, S=None, U=None, eps=1e-10):
        self.dim = dimension
        self.label = label
        self.A = np.asarray(A, dtype=float).copy()
        self.eps = float(eps)
        self.n = int(n)
        self.M = np.zeros(dimension, dtype=float) if M is None else np.asarray(M, dtype=float).copy()
        self.S = np.zeros((dimension, dimension), dtype=float) if S is None else np.asarray(S, dtype=float).copy()
        # U stores eigenvectors as columns.  This is equivalent to the reference
        # implementation's row-wise eig_component representation.
        self.U = np.eye(dimension, dtype=float) if U is None else np.asarray(U, dtype=float).copy()

    def __str__(self):
        return f"A node covering {self.n} points."


class VEBF:
    """Versatile Elliptic Basis Function classifier following the reference rules.

    Parameters
    ----------
    dimension : int
        Input dimensionality.
    merge_parameter : float
        Equivalent to the reference implementation's theta: overlap threshold
        used during same-class node merging.  Default is 0.
    eps : float
        Small numerical floor added when evaluating ellipsoids/eigenvalues.
    delta : float
        Scaling factor for the initial pairwise-distance width.  Default is 1.
    """

    # VEBF paper constant: width expansion starts only after the node has more
    # than two samples.
    N0 = 2

    def __init__(self, dimension, merge_parameter=0.0, eps=1e-10, delta=1.0):
        self.dim = int(dimension)
        self.theta = float(merge_parameter)
        self.eps = float(eps)
        self.delta = float(delta)

        self.nodes = {}
        self.init_width = {}
        self.classes_ = np.array([], dtype=object)

    # ------------------------------------------------------------------
    # Geometry helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _as_2d(X, dim=None):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if X.ndim != 2:
            raise ValueError(f"X must have shape (n_samples, n_features), got {X.shape}")
        if dim is not None and X.shape[1] != dim:
            raise ValueError(f"Expected {dim} features, got {X.shape[1]}")
        return X

    def average_pairwise_distance(self, X):
        """Reference VEBF initial-width formula.

        The reference implementation uses

            2 * delta / n^2 * sum_{i<j} ||x_i - x_j||_2

        and copies the resulting scalar into every axis.
        """
        X = self._as_2d(X, self.dim)
        n = len(X)
        distance = 0.0 if n < 2 else float(np.sum(pdist(X)))
        width = (2.0 * self.delta / (n ** 2)) * distance if n > 0 else 0.0
        return np.full(self.dim, width, dtype=float)

    def surface_single(self, x, U, A, k=1.0):
        """Reference L2 hyperellipsoidal function for one point."""
        x = np.asarray(x, dtype=float)
        U = np.asarray(U, dtype=float)
        A = np.asarray(A, dtype=float)
        x_local = U.T @ x
        margin = A + self.eps
        return np.linalg.norm(x_local / margin, ord=2) ** 2 - (k ** 2)

    def surface_multi(self, xs, Us, As, ks=None):
        """Reference L2 hyperellipsoidal function for point/node pairs."""
        xs = np.asarray(xs, dtype=float)
        Us = np.asarray(Us, dtype=float)
        As = np.asarray(As, dtype=float)

        if xs.ndim == 1:
            xs = xs[None, :]
        if Us.ndim != 3 or As.ndim != 2:
            raise ValueError("Us must have shape (n,d,d), As must have shape (n,d)")

        if ks is None:
            ks = np.ones(len(xs), dtype=float)
        else:
            ks = np.asarray(ks, dtype=float)
            if ks.ndim == 0:
                ks = np.full(len(xs), float(ks))

        # U has eigenvectors as columns, so U.T @ x is the local coordinate.
        xs_local = np.einsum("nij,nj->ni", Us.swapaxes(1, 2), xs)
        margin = As + self.eps
        return np.sum((xs_local / margin) ** 2, axis=1) - ks ** 2

    def _compute_sorted_eigencomponent(self, cov):
        """Match the reference eigendecomposition convention.

        Eigenvectors are returned as columns here, while the reference source
        stores them as rows.  The ordering and non-positive-eigenvalue handling
        follow the reference implementation.
        """
        cov = np.asarray(cov, dtype=float)
        eigen_value, eigen_vector = np.linalg.eig(cov)

        sort_idx = np.argsort(eigen_value)[::-1]
        eigen_value = eigen_value[sort_idx].real
        eigen_vector = eigen_vector[:, sort_idx].real

        first_zero_index = np.searchsorted(-eigen_value, 0)
        if first_zero_index < len(eigen_value):
            # Same lower-bound rule as the reference source.
            previous = eigen_value[first_zero_index - 1] if first_zero_index > 0 else eigen_value[-1]
            lowest_lambda = min(previous / 2.0, self.eps)
            eigen_value[first_zero_index:] = lowest_lambda

        return eigen_value, eigen_vector

    # ------------------------------------------------------------------
    # Initialization / public training API
    # ------------------------------------------------------------------
    def width_init(self, X, y):
        """Initialize widths for classes appearing in a batch."""
        X = self._as_2d(X, self.dim)
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of samples")

        new_classes = set(np.unique(y)) - set(self.init_width.keys())
        if new_classes:
            average_width = self.average_pairwise_distance(X)
            for label in new_classes:
                self.init_width[label] = average_width.copy()

    def _create_new_node(self, x, label):
        """Reference VEBF new-node initialization."""
        if label not in self.init_width:
            raise RuntimeError(
                "Initial width for this class is not initialized. "
                "Call fit/partial_fit with an initial batch before train(), "
                "or provide the class in a width_init() call."
            )

        x = np.asarray(x, dtype=float)
        return Node(
            dimension=self.dim,
            label=label,
            A=self.init_width[label],
            n=1,
            M=x,
            # The VEBF paper/reference starts a new node with zero covariance.
            S=np.zeros((self.dim, self.dim), dtype=float),
            U=np.eye(self.dim, dtype=float),
            eps=self.eps,
        )

    def fit(self, X, y, reset=True):
        """Fit on an initial batch, then process samples in streaming order.

        This mirrors the reference implementation's fit behavior: initial widths
        are computed from the complete batch before samples are consumed.
        """
        X = self._as_2d(X, self.dim)
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of samples")

        if reset:
            self.nodes = {}
            self.init_width = {}
            self.classes_ = np.array([], dtype=object)

        self.width_init(X, y)
        for x_i, y_i in zip(X, y):
            self._train_one(x_i, y_i)

        self._set_classes()
        return self

    def partial_fit(self, X, y):
        """Incrementally process a chunk without resetting existing state."""
        X = self._as_2d(X, self.dim)
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of samples")

        # The reference implementation initializes widths from the current
        # chunk for any classes that have not been seen before.
        self.width_init(X, y)
        for x_i, y_i in zip(X, y):
            self._train_one(x_i, y_i)

        self._set_classes()
        return self

    def train(self, x, y):
        """Backward-compatible one-sample training method.

        For exact reference-style initial-width behavior, initialize the model
        with fit() (or a multi-sample partial_fit()) first.  This method then
        processes one incoming sample at a time.
        """
        x = np.asarray(x, dtype=float)
        if x.ndim != 1 or x.shape[0] != self.dim:
            raise ValueError(f"x must have shape ({self.dim},)")
        self._train_one(x, y)
        self._set_classes()
        return self

    # ------------------------------------------------------------------
    # Online update
    # ------------------------------------------------------------------
    def _find_shortest(self, x, label):
        nodes = self.nodes[label]
        return min(nodes, key=lambda node: np.linalg.norm(node.M - x))

    def _train_one(self, x, label):
        x = np.asarray(x, dtype=float)
        if label not in self.nodes:
            self.nodes[label] = [self._create_new_node(x, label)]
            return

        # VEBF chooses the nearest center within the true class.
        node = self._find_shortest(x, label)

        n = node.n
        M = node.M
        S = node.S

        # Tentative center.
        M_new = (n * M + x) / (n + 1)

        # Tentative covariance, equivalent to the reference recurrence.
        xx = np.outer(x, x)
        MM = np.outer(M, M)
        MnewMnew = np.outer(M_new, M_new)
        S_new = (
            n * S + xx - MM
        ) / (n + 1) - MnewMnew + MM

        # Reference growth test uses the tentative covariance/eigenvectors but
        # the current node width.
        eigen_value, eigen_vector = self._compute_sorted_eigencomponent(S_new)
        psi_temp = self.surface_single(x - M_new, eigen_vector, node.A)

        if psi_temp > 0:
            # Incoming point cannot be reasonably learned by this node: create
            # a new same-class node and let it participate in merging.
            latest = self._create_new_node(x, label)
            self.nodes[label].append(latest)
            self._merge_node(latest)
            return

        # Update accepted into the current node.
        n_new = n + 1

        # Reference VEBF width update: no expansion until n > N0.
        if n_new > self.N0:
            displacement = M_new - M
            new_A = node.A + np.abs(eigen_vector.T @ displacement)
        else:
            new_A = node.A.copy()

        node.M = M_new
        node.U = eigen_vector
        node.S = S_new
        node.A = new_A
        node.n = n_new

        # The reference implementation performs at most one merge attempt for
        # the updated/created node.
        self._merge_node(node)

    # ------------------------------------------------------------------
    # Merge
    # ------------------------------------------------------------------
    def _merge_node(self, alpha):
        """Reference same-class merge rule; at most one merge per update."""
        nodes = self.nodes[alpha.label]
        if len(nodes) <= 1:
            return

        for beta in nodes:
            if beta is alpha:
                continue

            # Reference merge criterion: either center lies inside the other
            # node's ellipsoid.
            psi_alpha = self.surface_single(alpha.M - beta.M, beta.U, beta.A)
            psi_beta = self.surface_single(beta.M - alpha.M, alpha.U, alpha.A)

            if psi_alpha <= self.theta or psi_beta <= self.theta:
                n_gamma = alpha.n + beta.n
                M_gamma = (alpha.n * alpha.M + beta.n * beta.M) / n_gamma

                displacement = alpha.M - beta.M
                S_gamma = (
                    alpha.n * alpha.S
                    + beta.n * beta.S
                    + (alpha.n * beta.n / n_gamma) * np.outer(displacement, displacement)
                ) / n_gamma

                eigen_value, eigen_vector = self._compute_sorted_eigencomponent(S_gamma)
                A_gamma = np.sqrt(2.0 * math.pi * np.abs(eigen_value))

                beta.n = n_gamma
                beta.M = M_gamma
                beta.S = S_gamma
                beta.U = eigen_vector
                beta.A = A_gamma

                nodes.remove(alpha)
                return

    # ------------------------------------------------------------------
    # Prediction / utilities
    # ------------------------------------------------------------------
    def predict(self, X):
        X = self._as_2d(X, self.dim)
        nodes = self.get_nodes()
        if not nodes:
            raise AssertionError("Can't predict with empty VEBF.")

        centers = np.asarray([node.M for node in nodes])
        Us = np.asarray([node.U for node in nodes])
        As = np.asarray([node.A for node in nodes])

        distances = np.empty((len(X), len(nodes)), dtype=float)
        for i, x in enumerate(X):
            distances[i] = self.surface_multi(
                (x[None, :] - centers),
                Us,
                As,
            )

        nearest = np.argmin(distances, axis=1)
        labels = np.asarray([node.label for node in nodes], dtype=object)
        return labels[nearest]

    def get_nodes(self):
        return list(chain.from_iterable(self.nodes.values()))

    def find_shortest(self, x, y):
        return self._find_shortest(np.asarray(x, dtype=float), y)

    def find_shortest_all(self, x, nodes=None):
        x = np.asarray(x, dtype=float)
        nodes = self.get_nodes() if nodes is None else nodes
        return min(nodes, key=lambda node: np.linalg.norm(node.M - x))

    def _set_classes(self):
        self.classes_ = np.asarray(list(self.nodes.keys()))

    def __str__(self):
        total = sum(len(group) for group in self.nodes.values())
        string = f"A vebf with {total} nodes.\n"
        for label, nodes in self.nodes.items():
            for node in nodes:
                string += f"Label {label}: {node}\n"
        return string


# ----------------------------------------------------------------------
# Small self-checks for the reference rules.
# ----------------------------------------------------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(7)
    X0 = np.vstack([
        rng.normal(loc=(-2.0, 0.0), scale=0.25, size=(30, 2)),
        rng.normal(loc=(2.0, 0.0), scale=0.25, size=(30, 2)),
    ])
    y0 = np.array([0] * 30 + [1] * 30)

    model = VEBF(dimension=2)
    model.fit(X0, y0)

    pred = model.predict(X0)
    accuracy = float(np.mean(pred == y0))

    # Initial width check against the exact reference formula.
    expected_width = np.full(2, 2.0 * np.sum(pdist(X0)) / (len(X0) ** 2))
    for width in model.init_width.values():
        assert np.allclose(width, expected_width)

    # Basic geometry/state invariants.
    assert all(node.n >= 1 for node in model.get_nodes())
    assert all(node.A.shape == (2,) for node in model.get_nodes())
    assert all(node.U.shape == (2, 2) for node in model.get_nodes())
    assert model.classes_.size == 2

    print(model)
    print(f"Reference-rule self-check accuracy: {accuracy:.4f}")


A vebf with 2 nodes.
Label 0: A node covering 30 points.
Label 1: A node covering 30 points.

Reference-rule self-check accuracy: 1.0000
